In [1]:
from splinter import Browser
from bs4 import BeautifulSoup as soup 
import re
import pandas as pd
import numpy as np
import time
import json
import random

In [2]:
browser = Browser('chrome')
city = "Bangalore"
target_cars = 4000
cars_collected = 0
total_pages = 250 

In [3]:
def collect_car_links(city, total_pages, target_cars):

    cars_collected = 0

    with open(f"car_links_{city}.txt", "a") as f:

        for page_num in range(1, total_pages + 1):

            if page_num == 1:
                url = f"https://www.cardekho.com/used-cars+in+{city}"
            else:
                url = f"https://www.cardekho.com/used-cars+in+{city}/page-{page_num}"

            browser.visit(url)
            time.sleep(2)

            browser.execute_script("window.scrollTo(0, 1000);")
            time.sleep(1)

            current_soup = soup(browser.html, 'html.parser')

            page_links_found = 0

            for link in current_soup.find_all('a', href=True):
                href = link['href']

                if 'used-car-details' in href:

                    full_url = f"https://www.cardekho.com{href}" if href.startswith('/') else href

                    f.write(full_url + "\n")

                    cars_collected += 1
                    page_links_found += 1

            print(f"Page {page_num}: Saved {page_links_found} links. Total: {cars_collected}")

            if cars_collected >= target_cars:
                print("Target reached!")
                break

    print(f"All links are now saved in car_links_{city}.txt")

In [ ]:
collect_car_links(city, total_pages, target_cars)

## Main Extraction 

In [3]:
def scrape_car_links(city, links_filename="car_links.txt"):

    def get_browser():
        return Browser('chrome')

    # 1. Load links
    with open(links_filename, "r") as f:
        all_links = [line.strip() for line in f.readlines()]

    output_file = f"car_dataset_{city}.json"
    browser = get_browser()

    for index, link in enumerate(all_links):
        try:
            print(f"Scraping {index+1}/{len(all_links)}: {link}")

            # Visit page
            browser.visit(link)

            # Human delay + scroll
            time.sleep(random.uniform(1, 2))
            browser.execute_script("window.scrollTo(0, 600);")
            time.sleep(0.5)

            # Expand specifications
            try:
                view_all_spec_btn = browser.find_by_text('View all Specifications')
                if view_all_spec_btn:
                    browser.execute_script(
                        "arguments[0].click();",
                        view_all_spec_btn.first._element
                    )
                    print("Expanded specifications.")
                    time.sleep(0.6)
            except Exception:
                pass

            # Parse page
            page_soup = soup(browser.html, 'html.parser')

            car_data = {"url": link}

            # -------------------------
            # CAR NAME EXTRACTION
            # -------------------------
            name_tag = page_soup.find('div', class_='vehicleName')
            h1 = name_tag.find('h1') if (name_tag and name_tag.find('h1')) else page_soup.find('h1')

            if h1:
                parts = h1.get_text(separator="|", strip=True).split("|")
                car_data["car_name"] = parts[1].strip() if len(parts) >= 2 else parts[0].strip()

            # -------------------------
            # PRICE EXTRACTION
            # -------------------------
            price_div = page_soup.find('div', class_='vehiclePrice')
            if price_div:
                price_span = price_div.find('span')
                if price_span:
                    car_data["Price"] = price_span.get_text(strip=True)

            # -------------------------
            # SPECIFICATIONS EXTRACTION
            # -------------------------
            spec_items = page_soup.find_all('li', class_='gsc_col-xs-12')

            for item in spec_items:
                label_tag = item.find('div', class_='label')
                value_tag = item.find('span', class_='value-text')

                if label_tag and value_tag:
                    label = label_tag.get_text(strip=True)
                    value = value_tag.get_text(strip=True)
                    car_data[label] = value

            # Save data
            if len(car_data) > 1:
                with open(output_file, "a") as out:
                    out.write(json.dumps(car_data) + "\n")

                print(f"Saved: {car_data.get('Price','N/A')} and {len(car_data)-2} specs.")
            else:
                print(f"No data found for: {link}")

        except Exception as e:
            print(f"Serious error at {link}: {e}")

            browser.quit()
            browser = get_browser()
            time.sleep(1)
            continue

    browser.quit()

In [4]:
scrape_car_links("bangalore", "car_links_Bangalore.txt")

Scraping 1/4008: https://www.cardekho.com/used-car-details/used-Skoda-superb-laurin-klement-bsvi-cars-Bangalore_0f99bcae-34bd-4c54-ae3f-aa19e02e9cbc.htm?adId=19818&adType=41
Expanded specifications.
Saved: ₹26.65 Lakh and 53 specs.
Scraping 2/4008: https://www.cardekho.com/buy-used-car-details/used-Kia-seltos-gtx-cars-Bangalore_6a9bde96-1b2a-4f3e-8da3-87768b154eac.htm
Expanded specifications.
Saved: ₹10.37 Lakh and 42 specs.
Scraping 3/4008: https://www.cardekho.com/buy-used-car-details/used-Maruti-grand-vitara-alpha-at-bsvi-cars-Bangalore_44651c6f-a237-4463-911e-fe28ed962510.htm
Expanded specifications.
Saved: ₹12.82 Lakh and 41 specs.
Scraping 4/4008: https://www.cardekho.com/buy-used-car-details/used-Maruti-grand-vitara-delta-cars-Bangalore_8d13fab5-b2a3-4368-8379-553ea66234f3.htm
Expanded specifications.
Saved: ₹11.04 Lakh and 47 specs.
Scraping 5/4008: https://www.cardekho.com/used-car-details/used-Tata-safari-accomplished-cars-Bangalore_38bb74a0-f668-4562-b074-9228f1e15471.htm?ad

KeyboardInterrupt: 